# Main experiment — Kaggle 3 giai đoạn

Chạy **một dataset + một Stratified fold** mỗi phiên: Data Engineering → 12 baseline + TabNet → Performance Evaluation. Kết quả được ghi ngay sau từng model để resume; không lưu matrix trung gian, prediction từng mẫu hoặc model checkpoint.

## 1. Thư viện, CUDA và seed

In [ ]:
from pathlib import Path
import importlib.metadata as package_metadata
import gc, importlib.util, json, random, shutil, subprocess, sys, warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
KAGGLE_WORKING_ROOT = Path('/kaggle/working')
# /kaggle/working xác định runtime; /kaggle/input có thể rỗng khi chưa Add Input.
IS_KAGGLE = KAGGLE_WORKING_ROOT.is_dir()
if IS_KAGGLE and importlib.util.find_spec('pytorch_tabnet') is None:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'pytorch-tabnet==4.1.0'], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Hãy bật Internet trên Kaggle để cài pytorch-tabnet.') from exc

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    SMOTE = None
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    import torch
except ImportError:
    torch = None
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None
try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    from pytorch_tabnet.metrics import Metric as TabNetMetric
except ImportError:
    TabNetClassifier = TabNetMetric = None

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
RANDOM_STATE = SEED = 42
random.seed(SEED); np.random.seed(SEED)
CUDA_AVAILABLE = bool(torch is not None and torch.cuda.is_available())
if torch is not None:
    torch.manual_seed(SEED)
    if CUDA_AVAILABLE:
        torch.cuda.manual_seed_all(SEED)
print('Kaggle:', IS_KAGGLE, '| CUDA:', CUDA_AVAILABLE, '| GPU:', torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None)

## 2. Chọn dataset, fold và nhóm model

In [ ]:
# Cell này tự khởi tạo path để vẫn chạy đúng khi người dùng chạy lại riêng cell cấu hình.
from pathlib import Path
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
KAGGLE_WORKING_ROOT = Path('/kaggle/working')
IS_KAGGLE = KAGGLE_WORKING_ROOT.is_dir()
LOCAL_CWD = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (LOCAL_CWD, *LOCAL_CWD.parents) if (p / 'Notebook').is_dir()), LOCAL_CWD)
LOCAL_INPUT_ROOT = PROJECT_ROOT / 'data' / 'Raw_data'

DATASET_NAME = 'MLG_ULB'  # MLG_ULB | IEEE_CIS | SPARKOV
FOLD_ID = 0               # 0..4
N_SPLITS = 5
REMOVE_MLG_DUPLICATES = False
MAX_DENSE_GB = 16.0

ALL_MODELS = [
    'Logistic Regression', 'Decision Tree', 'Random Forest', 'LightGBM',
    'CatBoost', 'XGBoost', 'AdaBoost', 'TabNet', 'Extra Trees',
    'KNN', 'LDA', 'Naive Bayes', 'Gradient Boosting',
]
MODELS_TO_RUN = ALL_MODELS.copy()  # Có thể chia model qua nhiều session

DATA_HINTS = {
    'MLG_ULB': {'creditcard.csv': KAGGLE_INPUT_ROOT / 'mlg-ulb-creditcardfraud' / 'creditcard.csv'},
    'IEEE_CIS': {
        'train_transaction.csv': KAGGLE_INPUT_ROOT / 'ieee-fraud-detection' / 'train_transaction.csv',
        'train_identity.csv': KAGGLE_INPUT_ROOT / 'ieee-fraud-detection' / 'train_identity.csv',
    },
    'SPARKOV': {'fraudTrain.csv': KAGGLE_INPUT_ROOT / 'fraud-detection' / 'fraudTrain.csv'},
}
DATASET_SLUG = {'MLG_ULB': 'mlg_ulb', 'IEEE_CIS': 'ieee_cis', 'SPARKOV': 'sparkov'}
if DATASET_NAME not in DATA_HINTS or FOLD_ID not in range(N_SPLITS):
    raise ValueError('DATASET_NAME hoặc FOLD_ID không hợp lệ')
if not set(MODELS_TO_RUN).issubset(ALL_MODELS):
    raise ValueError('MODELS_TO_RUN có model ngoài phạm vi nghiên cứu')

def resolve_input_file(filename, hint):
    # 1) Đường dẫn Kaggle chuẩn/được gợi ý.
    if hint.is_file():
        return hint.resolve()
    # 2) Dò toàn bộ input đã được Add Input; local chỉ dò data/Raw_data.
    search_roots = []
    if KAGGLE_INPUT_ROOT.is_dir():
        search_roots.append(KAGGLE_INPUT_ROOT)
    if LOCAL_INPUT_ROOT.is_dir() and LOCAL_INPUT_ROOT not in search_roots:
        search_roots.append(LOCAL_INPUT_ROOT)
    matches = []
    for root in search_roots:
        matches.extend(root.rglob(filename))
    matches = list(dict.fromkeys(path.resolve() for path in matches if path.is_file()))
    if len(matches) == 1:
        return matches[0]
    if not matches:
        location = '/kaggle/input' if IS_KAGGLE else str(LOCAL_INPUT_ROOT)
        raise FileNotFoundError(
            f'Không tìm thấy {filename} trong {location}. '
            f'Nếu chạy Kaggle, hãy chọn Add Input và gắn dataset cho {DATASET_NAME}, rồi Restart & Run All.'
        )
    raise RuntimeError(f'Tìm thấy nhiều file tên {filename}; cần giữ đúng một input: {matches}')

INPUTS = {name: resolve_input_file(name, hint) for name, hint in DATA_HINTS[DATASET_NAME].items()}
RESULTS_ROOT = (KAGGLE_WORKING_ROOT if IS_KAGGLE else Path.cwd()) / 'results' / DATASET_SLUG[DATASET_NAME]
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_ROOT / f'fold_{FOLD_ID}_results.csv'
TABNET_CURVE_PATH = RESULTS_ROOT / f'fold_{FOLD_ID}_tabnet_curve.csv'
TABNET_FIGURE_PATH = RESULTS_ROOT / f'fold_{FOLD_ID}_tabnet_curve.png'
XAI_FOLD_ROOT = RESULTS_ROOT / 'xai_checkpoints' / f'fold_{FOLD_ID}'
MODEL_CHECKPOINT_DIR = XAI_FOLD_ROOT / 'models'
XAI_PREDICTION_DIR = XAI_FOLD_ROOT / 'validation_predictions'
PREPROCESSOR_PATH = XAI_FOLD_ROOT / 'preprocessor.joblib'
FEATURE_NAMES_PATH = XAI_FOLD_ROOT / 'feature_names.csv'
FOLD_INDICES_PATH = XAI_FOLD_ROOT / 'fold_indices.npz'
VALIDATION_REFERENCE_PATH = XAI_FOLD_ROOT / 'validation_reference.csv'
XAI_MANIFEST_PATH = XAI_FOLD_ROOT / 'manifest.json'
VALIDATION_MATRIX_PATH = XAI_FOLD_ROOT / 'validation_matrix.npz'
BACKGROUND_MATRIX_PATH = XAI_FOLD_ROOT / 'shap_background_matrix.npz'
BACKGROUND_REFERENCE_PATH = XAI_FOLD_ROOT / 'shap_background_reference.csv'
MODEL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
XAI_PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
# Resume qua Kaggle session: gắn output cũ làm input để kế thừa mọi fold và checkpoint.
if IS_KAGGLE and not any(RESULTS_ROOT.glob('fold_*_results.csv')):
    prior_result_dirs = sorted({p.parent for p in KAGGLE_INPUT_ROOT.rglob('fold_*_results.csv')
                                if p.parent.name == DATASET_SLUG[DATASET_NAME]})
    if len(prior_result_dirs) == 1:
        shutil.copytree(prior_result_dirs[0], RESULTS_ROOT, dirs_exist_ok=True)
        print('Resumed all folds/checkpoints from:', prior_result_dirs[0])
    elif len(prior_result_dirs) > 1:
        raise RuntimeError(f'Tìm thấy nhiều nguồn output resume cho {DATASET_NAME}: {prior_result_dirs}')
print('Runtime:', 'KAGGLE' if IS_KAGGLE else 'LOCAL')
print('Kaggle input root:', KAGGLE_INPUT_ROOT, '| exists:', KAGGLE_INPUT_ROOT.is_dir())
if KAGGLE_INPUT_ROOT.is_dir():
    print('Mounted inputs:', sorted(path.name for path in KAGGLE_INPUT_ROOT.iterdir()))
print('Inputs:', INPUTS); print('Output:', RESULTS_PATH); print('XAI checkpoints:', XAI_FOLD_ROOT); print('Models:', MODELS_TO_RUN)

## 3. EDA/schema gate tối thiểu

In [ ]:
required = {
    'creditcard.csv': {'Time', 'Amount', 'Class', *{f'V{i}' for i in range(1, 29)}},
    'train_transaction.csv': {'TransactionID', 'isFraud'},
    'train_identity.csv': {'TransactionID'},
    'fraudTrain.csv': {'trans_num', 'trans_date_trans_time', 'is_fraud'},
}
checks = []
for filename, path in INPUTS.items():
    preview = pd.read_csv(path, nrows=5)
    missing = sorted(required[filename] - set(preview.columns))
    checks.append({'file': filename, 'size_gb': path.stat().st_size / 1024**3, 'columns': len(preview.columns), 'schema_ok': not missing})
    if missing: raise ValueError(f'{filename} thiếu cột: {missing}')
display(pd.DataFrame(checks)); display(pd.read_csv(next(iter(INPUTS.values())), nrows=5))

## 4. Train-only preprocessing và SMOTE
Thứ tự cố định: Stratified split → fit preprocessing trên train → transform train/validation → SMOTE train 1:1. Validation luôn giữ phân phối thật.

In [ ]:
if SMOTE is None:
    raise ImportError('Thiếu imbalanced-learn. Hãy thêm/cài package này trong Kaggle image trước khi chạy preprocessing.')

def make_sparse_ohe():
    # sklearn >= 1.2 dùng sparse_output; image Kaggle cũ dùng sparse.
    kwargs = {'handle_unknown': 'ignore', 'dtype': np.float32}
    try:
        return OneHotEncoder(sparse_output=True, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=True, **kwargs)

def matrix_gb(x):
    if sparse.issparse(x): return (x.data.nbytes + x.indices.nbytes + x.indptr.nbytes) / 1024**3
    return np.asarray(x).nbytes / 1024**3

def matrix_finite(x):
    return bool(np.isfinite(x.data if sparse.issparse(x) else np.asarray(x)).all())

notes, drop_cols, cat_cols, num_cols = [], [], [], []
record_ids = None
if DATASET_NAME == 'MLG_ULB':
    raw = pd.read_csv(INPUTS['creditcard.csv'])
    duplicate_count = int(raw.duplicated().sum())
    if REMOVE_MLG_DUPLICATES: raw = raw.drop_duplicates().reset_index(drop=True)
    record_ids = pd.Series(raw.index.to_numpy(), index=raw.index, name='record_id')
    X_raw, y_raw = raw.drop(columns='Class'), raw['Class'].astype(np.int8)
    notes.append(f'duplicates={duplicate_count}; removed={REMOVE_MLG_DUPLICATES}')
elif DATASET_NAME == 'IEEE_CIS':
    tx = pd.read_csv(INPUTS['train_transaction.csv'])
    identity = pd.read_csv(INPUTS['train_identity.csv'])
    raw = tx.merge(identity, on='TransactionID', how='left', validate='one_to_one')
    assert len(raw) == len(tx)
    record_ids = raw['TransactionID'].copy().rename('record_id')
    X_raw, y_raw = raw.drop(columns=['isFraud', 'TransactionID']), raw['isFraud'].astype(np.int8)
    for col in [f'card{i}' for i in range(1, 7)]:
        if col in X_raw: X_raw[col] = X_raw[col].astype('string').fillna('Unknown').astype(object)
    notes.append('LEFT JOIN identity; TransactionID excluded; card1-card6 categorical')
    del tx, identity
else:
    raw = pd.read_csv(INPUTS['fraudTrain.csv'])
    raw.drop(columns=[c for c in ['Unnamed: 0'] if c in raw], inplace=True)
    dt = pd.to_datetime(raw['trans_date_trans_time'], errors='coerce')
    raw['transaction_hour'] = dt.dt.hour
    raw['transaction_day'] = dt.dt.day
    raw['transaction_month'] = dt.dt.month
    raw['transaction_weekday'] = dt.dt.dayofweek
    raw['is_weekend'] = (dt.dt.dayofweek >= 5).astype(np.int8)
    fixed_drop = ['trans_date_trans_time', 'trans_num', 'cc_num', 'first', 'last', 'street', 'dob']
    record_ids = raw['trans_num'].copy().rename('record_id')
    raw.drop(columns=[c for c in fixed_drop if c in raw], inplace=True)
    X_raw, y_raw = raw.drop(columns='is_fraud'), raw['is_fraud'].astype(np.int8)
    notes.append('time features; fixed identifier/high-cardinality fields excluded')

train_idx, valid_idx = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_raw, y_raw))[FOLD_ID]
assert not np.intersect1d(train_idx, valid_idx).size
if DATASET_NAME == 'IEEE_CIS':
    ratios = X_raw.iloc[train_idx].isna().mean()
    drop_cols = ratios[ratios > .50].index.tolist()
    X_raw = X_raw.drop(columns=drop_cols)
    notes.append(f'drop_missing_gt_50pct_from_train={len(drop_cols)}')

Xtr_raw, Xva_raw = X_raw.iloc[train_idx].copy(), X_raw.iloc[valid_idx].copy()
ytr_raw, y_valid = y_raw.iloc[train_idx].copy(), y_raw.iloc[valid_idx].to_numpy(np.int8)
if DATASET_NAME == 'MLG_ULB':
    prep = StandardScaler()
    Xtr = prep.fit_transform(Xtr_raw.astype(np.float32)).astype(np.float32)
    X_valid = prep.transform(Xva_raw.astype(np.float32)).astype(np.float32)
    feature_names = Xtr_raw.columns.astype(str).tolist()
    num_cols = feature_names.copy()
else:
    cat_cols = Xtr_raw.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    num_cols = [c for c in Xtr_raw.columns if c not in cat_cols]
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=-999)), ('scaler', StandardScaler())])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
                         ('onehot', make_sparse_ohe())])
    prep = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], sparse_threshold=1.0)
    numeric_dtypes = {col: np.float32 for col in num_cols}
    Xtr_raw = Xtr_raw.astype(numeric_dtypes)
    Xva_raw = Xva_raw.astype(numeric_dtypes)
    Xtr, X_valid = prep.fit_transform(Xtr_raw), prep.transform(Xva_raw)
    feature_names = prep.get_feature_names_out().astype(str).tolist()
    if sparse.issparse(Xtr): Xtr, X_valid = Xtr.tocsr().astype(np.float32), X_valid.tocsr().astype(np.float32)

assert int(ytr_raw.value_counts().min()) > 5
X_train, y_train = SMOTE(k_neighbors=5, sampling_strategy=1.0, random_state=42).fit_resample(Xtr, ytr_raw)
X_train = X_train.tocsr().astype(np.float32) if sparse.issparse(X_train) else np.asarray(X_train, np.float32)
y_train = np.asarray(y_train, np.int8)
gates = {
    'disjoint': not np.intersect1d(train_idx, valid_idx).size,
    'same_features': X_train.shape[1] == X_valid.shape[1],
    'finite_train': matrix_finite(X_train), 'finite_validation': matrix_finite(X_valid),
    'smote_1_to_1': len(np.unique(np.bincount(y_train))) == 1,
    'validation_original': len(y_valid) == len(valid_idx), 'validation_two_classes': set(np.unique(y_valid)) == {0, 1},
}
PREPROCESSING_VALIDATED = all(gates.values())
if not PREPROCESSING_VALIDATED: raise RuntimeError(gates)
display(pd.DataFrame({'gate': gates.keys(), 'passed': gates.values()}))
display(pd.DataFrame([{'dataset': DATASET_NAME, 'fold': FOLD_ID, 'train_before_smote': len(ytr_raw),
                       'train_after_smote': len(y_train), 'validation': len(y_valid), 'features': X_train.shape[1],
                       'train_gb': matrix_gb(X_train), 'validation_gb': matrix_gb(X_valid),
                       'validation_fraud_rate': y_valid.mean(), 'notes': '; '.join(notes)}]))

# Checkpoint nền tảng để tái tạo đúng input XAI trên validation thật.
joblib.dump(prep, PREPROCESSOR_PATH, compress=3)
pd.DataFrame({'feature_index': np.arange(len(feature_names)), 'feature_name': feature_names}).to_csv(FEATURE_NAMES_PATH, index=False)
np.savez_compressed(FOLD_INDICES_PATH, train_idx=np.asarray(train_idx), valid_idx=np.asarray(valid_idx))
validation_reference = pd.DataFrame({
    'validation_position': np.arange(len(valid_idx)),
    'source_index': np.asarray(valid_idx),
    'record_id': record_ids.iloc[valid_idx].to_numpy(),
    'y_true': y_valid,
})
validation_reference.to_csv(VALIDATION_REFERENCE_PATH, index=False)
validation_matrix_format = 'scipy_csr' if sparse.issparse(X_valid) else 'numpy_dense'
if sparse.issparse(X_valid):
    sparse.save_npz(VALIDATION_MATRIX_PATH, X_valid.tocsr(), compressed=True)
else:
    np.savez_compressed(VALIDATION_MATRIX_PATH, X_valid=np.asarray(X_valid, dtype=np.float32))

# Background SHAP lấy từ train thật trước SMOTE, tối đa 500 mẫu mỗi lớp.
background_positions = []
rng = np.random.default_rng(RANDOM_STATE)
ytr_array = ytr_raw.to_numpy(dtype=np.int8)
for class_value in (0, 1):
    candidates = np.flatnonzero(ytr_array == class_value)
    take = min(500, len(candidates))
    background_positions.extend(rng.choice(candidates, size=take, replace=False).tolist())
background_positions = np.asarray(sorted(background_positions), dtype=np.int64)
background_matrix = Xtr[background_positions]
background_matrix_format = 'scipy_csr' if sparse.issparse(background_matrix) else 'numpy_dense'
if sparse.issparse(background_matrix):
    sparse.save_npz(BACKGROUND_MATRIX_PATH, background_matrix.tocsr(), compressed=True)
else:
    np.savez_compressed(BACKGROUND_MATRIX_PATH, X_background=np.asarray(background_matrix, dtype=np.float32))
pd.DataFrame({
    'background_position': np.arange(len(background_positions)),
    'train_fold_position': background_positions,
    'source_index': np.asarray(train_idx)[background_positions],
    'record_id': record_ids.iloc[np.asarray(train_idx)[background_positions]].to_numpy(),
    'y_true': ytr_array[background_positions],
}).to_csv(BACKGROUND_REFERENCE_PATH, index=False)
def installed_version(package_name):
    try: return package_metadata.version(package_name)
    except package_metadata.PackageNotFoundError: return None

manifest = {
    'dataset': DATASET_NAME, 'fold_id': FOLD_ID, 'n_splits': N_SPLITS,
    'splitter': 'StratifiedKFold', 'shuffle': True, 'random_state': RANDOM_STATE,
    'target_distribution_validation': {str(k): int(v) for k, v in zip(*np.unique(y_valid, return_counts=True))},
    'smote': {'train_only': True, 'k_neighbors': 5, 'sampling_strategy': 1.0, 'random_state': RANDOM_STATE},
    'missing': {'numeric_fill': -999, 'categorical_fill': 'Unknown', 'ieee_drop_threshold': 0.50},
    'scaling': 'StandardScaler fit on train fold', 'encoding': 'OneHotEncoder fit on train fold when categorical exists',
    'drop_columns': drop_cols, 'numeric_columns': num_cols, 'categorical_columns': cat_cols,
    'feature_count': len(feature_names), 'decision_threshold': 0.5, 'notes': notes,
    'validation_matrix_format': validation_matrix_format,
    'background_matrix_format': background_matrix_format,
    'background_source': 'real train fold before SMOTE, up to 500 rows per class',
    'runtime': {'python': sys.version, 'packages': {
        name: installed_version(name) for name in
        ['numpy', 'pandas', 'scipy', 'scikit-learn', 'imbalanced-learn', 'torch',
         'pytorch-tabnet', 'xgboost', 'lightgbm', 'catboost']
    }},
}
with XAI_MANIFEST_PATH.open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)
print('Saved XAI preprocessing checkpoint:', XAI_FOLD_ROOT)
del raw, X_raw, y_raw, Xtr_raw, Xva_raw, ytr_raw, Xtr, prep, record_ids
gc.collect()

## 5. Resume và đánh giá dùng chung

In [ ]:
RESULT_COLUMNS = ['model', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'status']
DENSE_REQUIRED = {'TabNet', 'LDA', 'Naive Bayes', 'Gradient Boosting'}

def load_results():
    if not RESULTS_PATH.exists(): return pd.DataFrame(columns=RESULT_COLUMNS)
    df = pd.read_csv(RESULTS_PATH)
    for col in RESULT_COLUMNS:
        if col not in df: df[col] = np.nan
    return df[RESULT_COLUMNS]

def save_result(row):
    df = load_results(); df = df[df.model != row['model']]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    order = {name: i for i, name in enumerate(ALL_MODELS)}
    df['_order'] = df.model.map(order); df = df.sort_values('_order').drop(columns='_order')
    df.to_csv(RESULTS_PATH, index=False)
    print('Saved immediately:', RESULTS_PATH)

def model_slug(name):
    return name.lower().replace(' ', '_')

def model_checkpoint_path(name):
    suffix = '.zip' if name == 'TabNet' else '.joblib'
    return MODEL_CHECKPOINT_DIR / f'{model_slug(name)}{suffix}'

def prediction_checkpoint_path(name):
    return XAI_PREDICTION_DIR / f'{model_slug(name)}.csv'

def metadata_checkpoint_path(name):
    return MODEL_CHECKPOINT_DIR / f'{model_slug(name)}_metadata.json'

def checkpoint_complete(name):
    return model_checkpoint_path(name).is_file() and prediction_checkpoint_path(name).is_file() and metadata_checkpoint_path(name).is_file()

def is_completed(name):
    df = load_results()
    metric_completed = bool(((df.model == name) & (df.status == 'completed')).any())
    return metric_completed and checkpoint_complete(name)

def model_matrices(name):
    if name not in DENSE_REQUIRED or not sparse.issparse(X_train): return X_train, X_valid
    gb = (X_train.shape[0] + X_valid.shape[0]) * X_train.shape[1] * 4 / 1024**3
    if gb > MAX_DENSE_GB:
        raise MemoryError(f'{name} cần dense khoảng {gb:.2f} GiB > MAX_DENSE_GB={MAX_DENSE_GB}; không tự đổi preprocessing.')
    return X_train.toarray().astype(np.float32), X_valid.toarray().astype(np.float32)

def evaluate(model, X_eval):
    pred = np.asarray(model.predict(X_eval)).reshape(-1).astype(np.int8)
    if hasattr(model, 'predict_proba'): score = np.asarray(model.predict_proba(X_eval))[:, 1]
    else:
        raw_score = np.asarray(model.decision_function(X_eval)).reshape(-1)
        score = 1 / (1 + np.exp(-np.clip(raw_score, -30, 30)))
    p_curve, r_curve, _ = precision_recall_curve(y_valid, score)
    metrics = {'precision': precision_score(y_valid, pred, zero_division=0),
            'recall': recall_score(y_valid, pred, zero_division=0),
            'f1': f1_score(y_valid, pred, zero_division=0),
               'roc_auc': roc_auc_score(y_valid, score), 'pr_auc': auc(r_curve, p_curve)}
    return metrics, pred, score

def build_model(name, gpu=True):
    if name == 'Logistic Regression': return LogisticRegression(max_iter=1000, solver='saga', random_state=SEED)
    if name == 'Decision Tree': return DecisionTreeClassifier(random_state=SEED)
    if name == 'Random Forest': return RandomForestClassifier(n_estimators=100, max_depth=None, n_jobs=-1, random_state=SEED)
    if name == 'LightGBM':
        if LGBMClassifier is None: raise ImportError('lightgbm unavailable')
        return LGBMClassifier(n_estimators=200, learning_rate=.1, objective='binary', n_jobs=-1, random_state=SEED, verbosity=-1, **({'device_type': 'gpu'} if gpu and CUDA_AVAILABLE else {}))
    if name == 'CatBoost':
        if CatBoostClassifier is None: raise ImportError('catboost unavailable')
        return CatBoostClassifier(iterations=200, learning_rate=.1, loss_function='Logloss', verbose=False, random_seed=SEED, **({'task_type': 'GPU', 'devices': '0'} if gpu and CUDA_AVAILABLE else {}))
    if name == 'XGBoost':
        if XGBClassifier is None: raise ImportError('xgboost unavailable')
        kwargs = dict(n_estimators=200, learning_rate=.1, objective='binary:logistic', max_depth=6,
                      tree_method='hist', eval_metric='logloss', n_jobs=-1, random_state=SEED)
        try:
            xgb_major = int(package_metadata.version('xgboost').split('.')[0])
        except (package_metadata.PackageNotFoundError, ValueError):
            xgb_major = 2
        if xgb_major >= 2:
            kwargs['device'] = 'cuda' if gpu and CUDA_AVAILABLE else 'cpu'
        elif gpu and CUDA_AVAILABLE:
            kwargs['tree_method'] = 'gpu_hist'
        return XGBClassifier(**kwargs)
    if name == 'AdaBoost': return AdaBoostClassifier(n_estimators=100, learning_rate=.1, random_state=SEED)
    if name == 'Extra Trees': return ExtraTreesClassifier(n_estimators=100, max_depth=None, n_jobs=-1, random_state=SEED)
    if name == 'KNN': return KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    if name == 'LDA': return LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    if name == 'Naive Bayes': return GaussianNB()
    if name == 'Gradient Boosting': return GradientBoostingClassifier(n_estimators=100, learning_rate=.1, random_state=SEED)
    raise KeyError(name)

def save_xai_model_checkpoint(name, model, pred, score):
    checkpoint = model_checkpoint_path(name)
    if name == 'TabNet':
        base_path = checkpoint.with_suffix('')
        model.save_model(str(base_path))  # pytorch-tabnet tạo file .zip
    else:
        joblib.dump(model, checkpoint, compress=3)
    prediction_frame = validation_reference.copy()
    prediction_frame['y_pred'] = np.asarray(pred, dtype=np.int8)
    prediction_frame['y_score'] = np.asarray(score, dtype=np.float32)
    prediction_frame['case_type'] = np.select(
        [(prediction_frame.y_true == 1) & (prediction_frame.y_pred == 1),
         (prediction_frame.y_true == 0) & (prediction_frame.y_pred == 0),
         (prediction_frame.y_true == 0) & (prediction_frame.y_pred == 1),
         (prediction_frame.y_true == 1) & (prediction_frame.y_pred == 0)],
        ['TP', 'TN', 'FP', 'FN'], default='UNKNOWN')
    prediction_frame.to_csv(prediction_checkpoint_path(name), index=False)
    model_metadata = {
        'model': name, 'class': f'{type(model).__module__}.{type(model).__name__}',
        'checkpoint': checkpoint.name, 'dataset': DATASET_NAME, 'fold_id': FOLD_ID,
        'decision_threshold': 0.5, 'feature_count': int(X_train.shape[1]),
        'params_repr': repr(model.get_params(deep=False)) if hasattr(model, 'get_params') else '',
    }
    with metadata_checkpoint_path(name).open('w', encoding='utf-8') as handle:
        json.dump(model_metadata, handle, ensure_ascii=False, indent=2)
    if not checkpoint_complete(name):
        raise RuntimeError(f'Checkpoint XAI chưa đầy đủ cho {name}')

def load_model_checkpoint(name):
    checkpoint = model_checkpoint_path(name)
    if name == 'TabNet':
        if TabNetClassifier is None: raise ImportError('pytorch-tabnet unavailable')
        restored = TabNetClassifier()
        restored.load_model(str(checkpoint))
        return restored
    return joblib.load(checkpoint)

def failed_row(name, exc):
    print(f'{name} failed: {type(exc).__name__}: {exc}')
    save_result({'model': name, 'precision': np.nan, 'recall': np.nan, 'f1': np.nan, 'roc_auc': np.nan, 'pr_auc': np.nan, 'status': 'failed'})

print('Completed với checkpoint XAI đầy đủ:', [name for name in ALL_MODELS if is_completed(name)])

## 6. Chạy tuần tự 13 mô hình

In [ ]:
class TabNetF1Metric(TabNetMetric if TabNetMetric is not None else object):
    def __init__(self): self._name, self._maximize = 'f1', True
    def __call__(self, y_true, y_score): return f1_score(y_true, np.argmax(y_score, axis=1), zero_division=0)

for name in MODELS_TO_RUN:
    if is_completed(name):
        print(f'Skip {name}: already completed'); continue
    model = X_fit = X_eval = metrics = pred = score = None
    try:
        X_fit, X_eval = model_matrices(name)
        if name == 'TabNet':
            if torch is None or TabNetClassifier is None or TabNetMetric is None: raise ImportError('pytorch-tabnet unavailable')
            model = TabNetClassifier(
                n_d=64, n_a=64, lambda_sparse=1e-3, seed=SEED, mask_type='sparsemax',
                optimizer_fn=torch.optim.Adam, optimizer_params={'lr': .02},
                scheduler_fn=torch.optim.lr_scheduler.StepLR,
                scheduler_params={'step_size': 20, 'gamma': .9, 'is_batch_level': False},
                device_name='cuda' if CUDA_AVAILABLE else 'cpu', verbose=10)
            model.fit(X_fit, y_train, eval_set=[(X_fit, y_train), (X_eval, y_valid)],
                      eval_name=['train_smote', 'valid_real'], eval_metric=['auc', TabNetF1Metric],
                      loss_fn=torch.nn.functional.cross_entropy, max_epochs=100, patience=15,
                      batch_size=1024, virtual_batch_size=128, num_workers=0, drop_last=False)
        else:
            model = build_model(name, gpu=True)
            try:
                model.fit(X_fit, y_train)
            except Exception:
                if not (CUDA_AVAILABLE and name in {'LightGBM', 'CatBoost', 'XGBoost'}): raise
                del model; gc.collect(); torch.cuda.empty_cache()
                model = build_model(name, gpu=False); model.fit(X_fit, y_train)

        metrics, pred, score = evaluate(model, X_eval)
        save_xai_model_checkpoint(name, model, pred, score)

        if name == 'TabNet':
            history = pd.DataFrame({k: pd.Series(np.asarray(v)) for k, v in dict(model.history.history).items()})
            history.index.name = 'epoch'; history.reset_index().to_csv(TABNET_CURVE_PATH, index=False)
            fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
            for ax, (title, cols) in zip(axes, [('ROC-AUC', ['train_smote_auc', 'valid_real_auc']), ('F1', ['train_smote_f1', 'valid_real_f1'])]):
                for col in cols:
                    if col in history: ax.plot(history.index, history[col], label=col, linewidth=2)
                if getattr(model, 'best_epoch', None) is not None: ax.axvline(model.best_epoch, color='black', linestyle='--', alpha=.5)
                ax.set(title=title, xlabel='Epoch'); ax.grid(alpha=.25); ax.legend()
            fig.suptitle(f'TabNet train (SMOTE) vs validation (real) — {DATASET_NAME} fold {FOLD_ID}')
            plt.tight_layout(); fig.savefig(TABNET_FIGURE_PATH, dpi=160, bbox_inches='tight'); plt.show()

        row = {'model': name, **metrics, 'status': 'completed'}
        save_result(row)
        print(f"{name}: F1={row['f1']:.6f}, ROC-AUC={row['roc_auc']:.6f}, PR-AUC={row['pr_auc']:.6f}")
    except Exception as exc:
        failed_row(name, exc)
    finally:
        del model, X_fit, X_eval, metrics, pred, score
        gc.collect()
        if CUDA_AVAILABLE: torch.cuda.empty_cache()

## 7. Kết quả fold và tổng hợp 5-fold

In [ ]:
results = load_results()
display(results.sort_values(['status', 'f1'], ascending=[True, False]))
completed = {name for name in ALL_MODELS if is_completed(name)}
print(f'Completed {len(completed)}/13 | Missing:', [m for m in ALL_MODELS if m not in completed])

fold_paths = [RESULTS_ROOT / f'fold_{i}_results.csv' for i in range(N_SPLITS)]
available = [i for i, path in enumerate(fold_paths) if path.exists()]
print('Available folds:', available)
if available == list(range(N_SPLITS)):
    frames = [pd.read_csv(path).assign(fold=i) for i, path in enumerate(fold_paths)]
    all_folds = pd.concat(frames, ignore_index=True).query("status == 'completed'")
    counts = all_folds.groupby('model').fold.nunique()
    complete_models = counts[counts == N_SPLITS].index
    summary = all_folds[all_folds.model.isin(complete_models)].groupby('model')[['precision','recall','f1','roc_auc','pr_auc']].agg(['mean','std'])
    summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
    display(summary.reset_index().sort_values('f1_mean', ascending=False))
else:
    print('Chưa tổng hợp: cần đủ fold_0...fold_4.')

del X_train, X_valid, y_train, y_valid
gc.collect()
if CUDA_AVAILABLE: torch.cuda.empty_cache()

## Output được giữ

Mỗi fold giữ metrics, preprocessor, feature names, indices/reference, validation matrix, SHAP background từ train thật, checkpoint từng model và validation predictions có nhãn TP/TN/FP/FN. TabNet có thêm curve CSV/PNG. Không lưu SMOTE matrix vì XAI không được giải thích dữ liệu synthetic.